# ShiftOver · llm runtime

Standalone Local Jupyter / Colab / Kaggle launcher. Git is the source of truth. This
notebook runs **llm only**. Internal Python package and secret names
remain `nimbus` / `NIMBUS_*` for compatibility.

**Validation status:** structure, clean-clone safety, configuration compilation, and local
contract tests are validated. Model loading, GPU memory peaks, latency, and real media
quality are unverified until you run the smoke and representative samples on target GPUs.


## 1. Clean reset and sync

Edit the variables below. Reset can delete only this notebook's marked clone. Results stay outside it. Prefer an exact commit SHA.

In [ ]:
"""Standard-library-only clone/bootstrap helpers, also embedded in first cells.

The repository may be absent, so the first notebook cell cannot import this file.
The notebook generator copies these small helpers rather than inference code.
"""

import os
import shutil
import stat
import subprocess
import sys
import tempfile
from pathlib import Path
from urllib.parse import urlparse


def remove_owned_directory(target, parent):
    target, parent = Path(target), Path(parent).resolve()
    resolved = target.resolve()
    if target.is_symlink() or resolved.parent != parent:
        raise ValueError("Deletion target escaped the dedicated runtime root.")

    def remove_readonly(function, filename, exception):
        if not isinstance(exception[1], PermissionError):
            raise exception[1]
        path = Path(filename).resolve()
        if resolved not in path.parents:
            raise ValueError("Read-only cleanup target escaped the owned directory.")
        os.chmod(filename, stat.S_IREAD | stat.S_IWRITE)
        function(filename)

    # Git marks object files read-only on Windows. Only remove that attribute
    # inside the already verified disposable clone/environment.
    shutil.rmtree(resolved, onerror=remove_readonly)


def runtime_paths(environment=None):
    environment = environment or (
        "kaggle"
        if Path("/kaggle").exists()
        else "colab"
        if "COLAB_RELEASE_TAG" in os.environ
        else "local"
    )
    if environment == "kaggle":
        return Path("/kaggle/working/nimbus-runtime"), Path("/kaggle/working/nimbus-results")
    if environment == "colab":
        return Path("/content/nimbus-runtime"), Path("/content/drive/MyDrive/nimbus-results")
    return Path(
        tempfile.gettempdir()
    ) / "nimbus-notebook-runtime", Path.cwd() / "artifacts/notebook"


def stop_runtime_process(process, config=None):
    """Stop this notebook's controller and its owned workers before reset.

    Windows terminate() bypasses Python finally blocks, so stop the owned process
    tree explicitly. Linux uses the controller's cooperative signal handler.
    """
    if process.poll() is not None:
        return process.returncode
    if os.name == "nt":
        subprocess.run(
            ["taskkill", "/PID", str(process.pid), "/T", "/F"],
            check=True,
            stdout=subprocess.DEVNULL,
        )
    else:
        process.terminate()
    code = process.wait(timeout=45)
    if os.name == "nt" and config and config.get("run_id"):
        import json

        manifest = Path(config["results_dir"]) / config["run_id"] / "run.json"
        if manifest.is_file():
            metadata = json.loads(manifest.read_text(encoding="utf-8"))
            if metadata["status"] in {"starting", "running", "resolving_models"}:
                metadata.update(status="stopped", stop_method="windows_owned_process_tree")
                manifest.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return code


def sync_repository(repo_url, git_ref, runtime_root, results_dir, clean_reset=True):
    runtime_root, results_dir = Path(runtime_root).resolve(), Path(results_dir).resolve()
    source = runtime_root / "source"
    if source == Path.cwd().resolve() or source in Path.cwd().resolve().parents:
        raise ValueError("The disposable clone cannot contain the working notebook directory.")
    if results_dir == source or source in results_dir.parents:
        raise ValueError("Results must live outside the disposable source clone.")
    parsed = urlparse(str(repo_url))
    if parsed.username or parsed.password:
        raise ValueError("Do not embed a Git credential in the repository URL.")
    if not git_ref or git_ref.startswith("-"):
        raise ValueError("Choose a branch, tag, or commit SHA.")
    runtime_root.mkdir(parents=True, exist_ok=True)
    marker = source / ".nimbus-runtime-clone"
    if source.exists() and clean_reset:
        if (
            source.is_symlink()
            or not marker.is_file()
            or marker.read_text(encoding="utf-8").strip() != str(repo_url)
        ):
            raise ValueError(
                "Refusing to reset a directory without this notebook's clone ownership marker."
            )
        if source.resolve().parent != runtime_root:
            raise ValueError("Clone reset target escaped the dedicated runtime root.")
        remove_owned_directory(source, runtime_root)
    environment = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if not source.exists():
        subprocess.run(
            ["git", "clone", "--no-hardlinks", "--no-checkout", "--", str(repo_url), str(source)],
            check=True,
            env=environment,
        )
        marker.write_text(str(repo_url), encoding="utf-8")
    elif not marker.exists() or marker.read_text(encoding="utf-8").strip() != str(repo_url):
        raise ValueError("Existing source directory is not the requested disposable clone.")
    subprocess.run(["git", "fetch", "origin", git_ref], cwd=source, check=True, env=environment)
    subprocess.run(
        ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=source, check=True, env=environment
    )
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip()
    if not (source / "pyproject.toml").exists() or not (source / "src/nimbus/runtime.py").exists():
        raise ValueError(
            "This commit lacks the NIMBUS implementation. Commit and push the local source first."
        )
    results_dir.mkdir(parents=True, exist_ok=True)
    return {"repo_dir": str(source), "results_dir": str(results_dir), "git_commit": commit}


def install_environment(repo_dir, runtime_root, profile, llm_backend="vllm", reset=False):
    repo_dir, runtime_root = Path(repo_dir).resolve(), Path(runtime_root).resolve()
    name = "media" if profile in {"speech", "handwriting", "media"} else "llm"
    environment = runtime_root / f"venv-{name}-{llm_backend if name == 'llm' else 'torch'}"
    marker = environment / ".nimbus-worker-env"
    if reset and environment.exists():
        if (
            environment.is_symlink()
            or environment.resolve().parent != runtime_root
            or not marker.is_file()
        ):
            raise ValueError("Refusing to reset an environment outside the owned runtime root.")
        remove_owned_directory(environment, runtime_root)
    if not environment.exists():
        # Media and transformer fallbacks inherit the platform's CUDA-enabled
        # torch. vLLM uses an isolated environment with its own matched stack.
        arguments = [sys.executable, "-m", "venv"]
        if name == "media" or llm_backend == "transformers":
            arguments.append("--system-site-packages")
        subprocess.run([*arguments, str(environment)], check=True)
        marker.write_text("nimbus", encoding="utf-8")
    executable = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    subprocess.run([str(executable), "-m", "pip", "install", "--upgrade", "pip"], check=True)
    # CPU notebook users without any preinstalled torch get an explicit CPU wheel.
    torch_present = (
        subprocess.run(
            [str(executable), "-c", "import torch"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        ).returncode
        == 0
    )
    if not torch_present:
        if name == "media" or llm_backend == "transformers":
            subprocess.run(
                [
                    str(executable),
                    "-m",
                    "pip",
                    "install",
                    "torch",
                    "--index-url",
                    "https://download.pytorch.org/whl/cpu",
                ],
                check=True,
            )
    profile_file = (
        "remote-media-cuda.txt"
        if name == "media" or llm_backend == "transformers"
        else "remote-llm-cuda.txt"
    )
    subprocess.run(
        [
            str(executable),
            "-m",
            "pip",
            "install",
            "-e",
            str(repo_dir),
            "-r",
            str(repo_dir / "requirements" / profile_file),
        ],
        check=True,
    )
    subprocess.run([str(executable), "-m", "pip", "check"], check=True)
    return str(executable)


def fetch_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if Path("/kaggle").exists():
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import userdata

        return userdata.get(name)
    raise ValueError(f"Set environment variable {name} for worker mode.")


def save_dependency_snapshot(executable, destination):
    output = subprocess.check_output([executable, "-m", "pip", "freeze"], text=True)
    Path(destination).write_text(output, encoding="utf-8")
    return {"snapshot": str(destination), "lines": len(output.splitlines())}


# Configuration: change these variables, then run top to bottom.
import json
REPO_URL = "https://github.com/saltypal/NIMBUS.git"
GIT_REF = "main"  # Prefer the exact 40-character commit SHA you reviewed.
WORKLOAD = "llm"  # this notebook runs only: llm
MODE = "worker"  # worker pulls real jobs; smoke only loads/tests the selected adapter(s)
BACKEND = "auto"  # auto | cuda | cpu; TPU adapters are not implemented
LLM_BACKEND = "auto"  # relevant only to the llm notebook
PIPELINE_PARALLEL_SIZE = None  # llm auto: 2 on dual GPUs, otherwise 1
CLEAN_RESET = True  # replaces only the dedicated, marked disposable clone
RESET_ENVIRONMENT = False  # True rebuilds only the marked worker venv
SEED = 42
BRIDGE_URL = ""  # HTTPS tunnel to restricted bridge :8001; worker mode only
MODEL_IDS = {
    "llm": "Qwen/Qwen3-8B"
}
MODEL_REVISIONS = {
    "llm": "b968826d9c46dd6066d109eabc6255188de91218"
}  # immutable HF commit SHAs
SMOKE_INPUTS = {}  # optional e.g. {"speech": "/kaggle/input/sample/note.wav"}

# Optional end-to-end API test. Leave empty on Kaggle and use the laptop browser Test tab.
# Kaggle cannot reach http://127.0.0.1 on your laptop. Configure only a private,
# authenticated Tailscale/SSH endpoint for the core app; never publish MongoDB.
APP_API_URL = ""
TEST_SOURCE_TYPE = "text"  # audio | handwriting; text does not queue a worker
TEST_TEXT = "Pump P-12 vibration increased after tool change."
TEST_AUDIO_PATH = ""
TEST_HANDWRITING_PATH = ""
TEST_TIMEOUT_SECONDS = 1200
ALLOWED_TEST_SOURCE_TYPES = []

RUNTIME_ROOT, RESULTS_DIR = runtime_paths()
if "COLAB_RELEASE_TAG" in os.environ and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if "runtime_process" in globals() and runtime_process.poll() is None:
    stop_runtime_process(runtime_process, globals().get("config"))
source = sync_repository(REPO_URL, GIT_REF, RUNTIME_ROOT, RESULTS_DIR, CLEAN_RESET)
print(json.dumps(source, indent=2))
if len(GIT_REF) != 40:
    print("NOTE: the moving ref resolved to the exact commit above; pin that SHA for reruns.")


## 2. Install the isolated worker environment

The app environment and GPU environment are separate. Installation ends with `pip check` and records an exact dependency snapshot.

In [ ]:

# Install outside the notebook kernel so model libraries cannot destabilize it.
import platform
cuda_available = False
gpu_capabilities = []
try:
    import torch
    cuda_available = torch.cuda.is_available()
    if cuda_available:
        gpu_capabilities = [
            torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())
        ]
except ImportError:
    pass
resolved_llm_backend = LLM_BACKEND
if resolved_llm_backend == "auto":
    suitable = (BACKEND != "cpu" and cuda_available and platform.system() == "Linux"
                and all(cap >= (7, 5) for cap in gpu_capabilities))
    resolved_llm_backend = "vllm" if suitable else "transformers"
worker_python = install_environment(
    source["repo_dir"], RUNTIME_ROOT, WORKLOAD, resolved_llm_backend, RESET_ENVIRONMENT
)
snapshot = RESULTS_DIR / f"dependencies-{WORKLOAD}-{source['git_commit'][:12]}.txt"
print(save_dependency_snapshot(worker_python, snapshot))


## 3. Inspect hardware and preflight the model

This checks CUDA capability and conservative RAM/VRAM estimates. A pass does not prove the real model fits.

In [ ]:

# Inspect the exact worker interpreter and reject obvious hardware/model mismatches
# before any weights are downloaded. Passing is an estimate; the real load remains required.
probe_code = ("import json; from nimbus.runtime import environment_info; "
              "print(json.dumps(environment_info()))")
probe = subprocess.check_output([worker_python, "-c", probe_code], text=True)
hardware = json.loads(probe)
print(json.dumps(hardware, indent=2))
config = {
    **source, "workload": WORKLOAD, "mode": MODE, "backend": BACKEND,
    "llm_backend": resolved_llm_backend, "models": MODEL_IDS, "revisions": MODEL_REVISIONS,
    "inputs": SMOKE_INPUTS, "seed": SEED, "bridge_url": BRIDGE_URL,
    "max_model_len": 8192, "llm_port": 9000,
}
from datetime import datetime, timezone
from uuid import uuid4
config["run_id"] = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
if PIPELINE_PARALLEL_SIZE is not None:
    config["pipeline_parallel_size"] = PIPELINE_PARALLEL_SIZE
config_path = RESULTS_DIR / f"runtime-config-{WORKLOAD}-{config['run_id']}.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
layout_code = ("import json,sys; from nimbus.runtime import choose_layout; "
               "print(json.dumps(choose_layout(json.loads(sys.argv[1]), "
               "json.loads(sys.argv[2])), indent=2))")
subprocess.run([worker_python, "-c", layout_code, json.dumps(config), json.dumps(hardware)],
               check=True)


## 4. Launch

Workers connect outward to bridge port 8001. No notebook port or MongoDB connection is exposed. The LLM server binds runtime loopback only.

In [ ]:

# Keys are read from runtime secrets and never written to config, output, or Git.
worker_environment = os.environ.copy()
if MODE == "worker":
    if not BRIDGE_URL:
        raise ValueError("Set BRIDGE_URL to restricted bridge :8001, or choose MODE='smoke'.")
    selected = ["speech", "handwriting"] if WORKLOAD == "media" else [WORKLOAD]
    for capability in selected:
        name = f"NIMBUS_BRIDGE_{capability.upper()}_KEY"
        worker_environment[name] = fetch_secret(name)
controller_log = RESULTS_DIR / f"controller-{WORKLOAD}-{config['run_id']}.log"
log_handle = controller_log.open("w", encoding="utf-8")
command = [worker_python, "-m", "nimbus.cli", "runtime", "--config", str(config_path)]
runtime_process = subprocess.Popen(
    command, cwd=source["repo_dir"], env=worker_environment,
    stdout=log_handle, stderr=subprocess.STDOUT,
    creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
)
log_handle.close()
print({"controller_pid": runtime_process.pid, "log": str(controller_log), "mode": MODE})


## 5. Inspect bounded logs

Workers register only after adapter smoke. Synthetic smoke proves execution, not production accuracy.

In [ ]:

# Rerun while the controller runs. Weight downloads and first load can take several minutes.
print({"controller_exit_code": runtime_process.poll()})
print(controller_log.read_text(encoding="utf-8", errors="replace")[-8000:])
for log_path in sorted(RESULTS_DIR.glob("*/**/*.log"))[-6:]:
    print(log_path.name, log_path.read_text(encoding="utf-8", errors="replace")[-2500:])


## 6. Optional end-to-end product test

The preferred path is the local ShiftOver browser Test tab. Configure `APP_API_URL` only when the core app is reachable through a private authenticated tunnel.

In [ ]:

# Optional full path: notebook -> private app API -> local queue -> outbound worker pull -> result.
# The normal setup needs no app tunnel: leave APP_API_URL empty and submit in the laptop Test tab.
if not ALLOWED_TEST_SOURCE_TYPES:
    print("The inference-test endpoint does not queue an LLM job. Validate the LLM through "
          "ShiftOver's normal local draft flow while this worker runs.")
elif not APP_API_URL:
    print("APP_API_URL is empty. Use ShiftOver's local browser Test tab while this worker runs.")
else:
    from urllib.parse import urlparse
    parsed_api = urlparse(APP_API_URL)
    if Path("/kaggle").exists() or "COLAB_RELEASE_TAG" in os.environ:
        if parsed_api.hostname in {"127.0.0.1", "localhost", "::1"}:
            raise ValueError("Notebook localhost is not the laptop. Use a private authenticated URL.")
        if parsed_api.scheme != "https":
            raise ValueError("Remote APP_API_URL must be private authenticated HTTPS.")
    if TEST_SOURCE_TYPE not in ALLOWED_TEST_SOURCE_TYPES:
        raise ValueError(f"Choose one of {ALLOWED_TEST_SOURCE_TYPES} for this worker profile.")
    test_file = (TEST_AUDIO_PATH if TEST_SOURCE_TYPE == "audio"
                 else TEST_HANDWRITING_PATH)
    if not test_file or not Path(test_file).is_file():
        raise ValueError(f"Set an existing local path for {TEST_SOURCE_TYPE} before testing.")
    api_test_code = r"""
import json, pathlib, sys, time
import httpx
base, source_type, text, filename, timeout_text = sys.argv[1:]
data = {"source_type": source_type}
if text:
    data["text"] = text
files = None
handle = None
try:
    if filename:
        path = pathlib.Path(filename)
        handle = path.open("rb")
        files = {"file": (path.name, handle, "application/octet-stream")}
    with httpx.Client(timeout=60, trust_env=False) as client:
        response = client.post(base.rstrip("/") + "/api/inference-tests", data=data, files=files)
        response.raise_for_status()
        state = response.json()
        test_id = state.get("id") or state.get("_id")
        if not test_id:
            raise RuntimeError("Inference-test response did not include an id")
        deadline = time.monotonic() + int(timeout_text)
        while state["status"] in {"queued", "leased"} and time.monotonic() < deadline:
            time.sleep(3)
            poll = client.get(base.rstrip("/") + f"/api/inference-tests/{test_id}")
            poll.raise_for_status()
            state = poll.json()
            print(json.dumps(state, indent=2))
        if state["status"] != "succeeded":
            raise RuntimeError(f"End-to-end inference test ended as {state['status']}: {state.get('error')}")
        print(json.dumps({"verified_status": state["status"], "job_id": state.get("job_id"),
                          "capability": state.get("capability"), "worker_id": state.get("worker_id"),
                          "model_id": state.get("model_id"),
                          "model_revision": state.get("model_revision"),
                          "transcript": state.get("transcript")}, indent=2))
finally:
    if handle:
        handle.close()
"""
    subprocess.run([worker_python, "-c", api_test_code, APP_API_URL, TEST_SOURCE_TYPE,
                    TEST_TEXT if TEST_SOURCE_TYPE == "text" else "", test_file,
                    str(TEST_TIMEOUT_SECONDS)], check=True)


## 7. Stop and retain results

Stop owned processes and preserve the manifest, exact source/model revisions, dependency snapshot, and logs.

In [ ]:

# Explicit shutdown, then create a portable archive before the temporary runtime is removed.
if runtime_process.poll() is None:
    stop_runtime_process(runtime_process, config)
print({"controller_exit_code": runtime_process.poll()})
archive_path = RESULTS_DIR.parent / f"shiftover-{WORKLOAD}-{config['run_id']}-results"
archive = shutil.make_archive(str(archive_path), "zip", RESULTS_DIR)
print({"results_archive": archive})
if Path("/kaggle").exists():
    print("Save a Kaggle version with outputs or download this archive before session reset.")
